In [1]:
import numpy as np
from collections import Counter
from sklearn.cluster import KMeans
from scipy.io import loadmat
from scipy.spatial.distance import cdist

In [ ]:
data = loadmat("data/mnist_10digits.mat")
xtrain = data['xtrain']
ytrain = data['ytrain'].flatten()

print("xtrain shape:", xtrain.shape)
print("ytrain shape:", ytrain.shape)

xtrain shape: (60000, 784)
ytrain shape: (60000,)


In [3]:
def compute_purity(labels, true_labels, n_clusters=10):
    purity_results = []
    for i in range(n_clusters):
        indices = np.where(labels == i)[0]
        if len(indices) == 0:
            purity_results.append({"true_label": None, "purity_score": 0})
            continue
        cluster_labels = true_labels[indices]
        most_common_label, count = Counter(cluster_labels).most_common(1)[0]
        purity = count / len(indices)
        purity_results.append({"true_label": int(most_common_label), "purity_score": purity})
    return purity_results


In [4]:
from sklearn.cluster import KMeans

x_norm = xtrain / 255.0
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
cluster_labels_euclidean = kmeans.fit_predict(x_norm)

purity_euclidean = compute_purity(cluster_labels_euclidean, ytrain)

# Display results
print("Purity using Euclidean Distance:")
for i, result in enumerate(purity_euclidean):
    print(f"Cluster {i}: True Label = {result['true_label']}, Purity = {result['purity_score']:.4f}")


Purity using Euclidean Distance:
Cluster 0: True Label = 1, Purity = 0.5276
Cluster 1: True Label = 2, Purity = 0.8972
Cluster 2: True Label = 1, Purity = 0.6239
Cluster 3: True Label = 8, Purity = 0.5311
Cluster 4: True Label = 0, Purity = 0.9073
Cluster 5: True Label = 7, Purity = 0.4267
Cluster 6: True Label = 3, Purity = 0.5281
Cluster 7: True Label = 0, Purity = 0.7808
Cluster 8: True Label = 4, Purity = 0.3572
Cluster 9: True Label = 6, Purity = 0.8592


In [5]:
def hamming_kmeans(X, k, max_iter=10):
    np.random.seed(42)
    centroids_idx = np.random.choice(X.shape[0], k, replace=False)
    centroids = X[centroids_idx]

    for iteration in range(max_iter):
        distances = cdist(X, centroids, metric='hamming')
        labels = np.argmin(distances, axis=1)

        new_centroids = np.zeros_like(centroids)
        for i in range(k):
            members = X[labels == i]
            if members.shape[0] > 0:
                new_centroids[i] = np.round(np.mean(members, axis=0))
            else:
                new_centroids[i] = centroids[i]
        centroids = new_centroids

    return labels, centroids


In [6]:
# Thresholding the image pixels
x_binary = (xtrain > 128).astype(int)

# Apply Hamming-based k-means
cluster_labels_hamming, centroids_hamming = hamming_kmeans(x_binary, k=10)

purity_hamming = compute_purity(cluster_labels_hamming, ytrain)

# Display results
print("Purity using Hamming Distance:")
for i, result in enumerate(purity_hamming):
    print(f"Cluster {i}: True Label = {result['true_label']}, Purity = {result['purity_score']:.4f}")


Purity using Hamming Distance:
Cluster 0: True Label = 5, Purity = 0.1873
Cluster 1: True Label = 3, Purity = 0.5637
Cluster 2: True Label = 8, Purity = 0.3676
Cluster 3: True Label = 1, Purity = 0.5421
Cluster 4: True Label = 0, Purity = 0.7663
Cluster 5: True Label = 6, Purity = 0.6325
Cluster 6: True Label = 7, Purity = 0.4960
Cluster 7: True Label = 1, Purity = 0.4177
Cluster 8: True Label = 0, Purity = 0.8743
Cluster 9: True Label = 4, Purity = 0.4675


In [7]:
# Compare average purities
avg_purity_euclidean = np.mean([p['purity_score'] for p in purity_euclidean])
avg_purity_hamming = np.mean([p['purity_score'] for p in purity_hamming])

print(f"\nAverage Purity - Euclidean: {avg_purity_euclidean:.4f}")
print(f"Average Purity - Hamming  : {avg_purity_hamming:.4f}")

if avg_purity_euclidean > avg_purity_hamming:
    print("Euclidean distance provides better clustering purity on MNIST.")
else:
    print("Hamming distance provides better clustering purity on MNIST.")



Average Purity - Euclidean: 0.6439
Average Purity - Hamming  : 0.5315
Euclidean distance provides better clustering purity on MNIST.
